# 01 — Data Exploration

Profiles the raw synthetic wheat dataset and checks the assumptions the
rest of the pipeline relies on: one row per group per day, a complete
`modal_price`, and known missingness in `min_price`, `max_price` and
`arrivals_tonnes`.

Run from the `ml/notebooks/` directory.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 60)


In [ ]:
from src import data_loader, validation, preprocessing

raw = data_loader.load_raw()
df = preprocessing.clean(preprocessing.coerce_numeric(raw))
validation.validate_schema(df)
df.head()

## Quality report

The same report `python -m src.preprocessing` writes to `data/reports/data_quality.json`.

In [ ]:
report = validation.quality_report(df)
for key, value in report.items():
    print(f'{key}: {value}')

## Price level by mandi

Different mandis sit at different price levels — the reason the model learns the *change* rather than the level.

In [ ]:
df.groupby('mandi')['modal_price'].describe()[['count', 'mean', 'std', 'min', 'max']]

## Year-on-year drift

If the level drifts upward, a tree trained on 2021-23 levels cannot extrapolate into 2025.

In [ ]:
df.assign(year=df['date'].dt.year).groupby(['year'])['modal_price'].mean().round(2)

## Day-to-day movement

How much does the price actually move? This sets the floor on achievable MAE.

In [ ]:
moves = df.sort_values(['mandi', 'variety', 'grade', 'date']).groupby(
    ['mandi', 'variety', 'grade']
)['modal_price'].diff()
moves.abs().describe().round(3)